# ML1 - GridSearch with DecisionTreeRegressor, RandomForestRegressor

This notebook follows the ML1_Pipeline_Model notebook and focuses on:
1. Using DecisionTreeRegressor and RandomForestRegressor (or others)  
2. Hyperparameter tuning with GridSearchCV  
3. Cross-validation

## 1. Importing Libraries

In [ ]:
# Necessary libraries.
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import cross_validate
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

## 2. Loading and Preparing Data with the Pipeline

We follow the same preparation steps as in the previous notebook.


In [ ]:
# Load the data from the pickle file.
df = pd.read_pickle('arbres_clean_Laura.pkl')

X = df.drop(columns=['anneedeplantation'])
y = df['anneedeplantation']
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=20)
categorical_variables = X_train.select_dtypes(include = ['str']).columns.tolist()
numerical_variables = X_train.select_dtypes(include = ['number']).columns.tolist()
print(categorical_variables,numerical_variables)
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))])

numerical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())])
preprocessor = ColumnTransformer([
    ('cat', categorical_pipeline, categorical_variables),
    ('num', numerical_pipeline, numerical_variables)])
X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

In [ ]:
#récupération du df de travail, qu'on va appeler arbres
# X_transformed = preprocessor.fit_transform(df)

# arbres = pd.DataFrame.sparse.from_spmatrix(
#     X_transformed,
#     columns=preprocessor.get_feature_names_out(),
#     index=df.index
# )

# arbres.sample(10)

## 3. Initial Cross-Validation

Before tuning hyperparameters, let's evaluate a model with default parameters using cross-validation.


In [ ]:
model = Pipeline([('preprocessor', preprocessor),
                ('regressor', LinearRegression())])

scoring = ['neg_mean_absolute_error', 
           'neg_root_mean_squared_error', 
           'r2']

cv_results = cross_validate(
    model,
    X_train,
    y_train,
    cv=5,
    scoring=scoring)

mae_scores = -cv_results['test_neg_mean_absolute_error']
rmse_scores = -cv_results['test_neg_root_mean_squared_error']
r2_scores = cv_results['test_r2']

print(f"MAE: {mae_scores.mean():.2f} ± {mae_scores.std():.2f}")
print(f"RMSE: {rmse_scores.mean():.2f} ± {rmse_scores.std():.2f}")
print(f"R2: {r2_scores.mean():.3f} ± {r2_scores.std():.3f}")

## 4. 🌳 Modeling with GridSearchCV

Let's choose two machine learning models.

Evaluate the models with hyperparameter tuning (testing different hyperparameter values).  
Optimize the models as you wish, for example for Decision Tree and Random Forest:

### 🔍 Hyperparameters Tested
- `max_depth`: maximum depth of the trees [3, 4, 5, 10]  
- `min_samples_leaf`: minimum number of samples required at a leaf [0.1, 0.05]  
- `n_estimators` (for Random Forest only): number of trees in the forest [50, 100, 200]

Let's use GridSearchCV to automate the search for the best hyperparameters.

In [ ]:
# Hyperparameter tuning with GridSearchCV, and we can print the MSE or MAE scores for each cross-validation fold.
rf = RandomForestRegressor(random_state=20)


## 5. Final Evaluation on the Test Set

In [ ]:
# Code for evaluating the best model on the test set.

## 6. Visualization of Predictions  
One model will suffice.

In [ ]:
# Visualisation of predictions.

## [OPTIONAL] 7.Feature Importance

Using the random forest, how can we determine which features are the most important?

In [ ]:
# Code. In your opinion, how is feature importance calculated in Random Forest?

Typically, one might consider using backward elimination on feature importance to select the best subset of features, but this can be time-consuming if there are many features and the model takes a while to train, as is the case here. Instead, we can use Lasso-regularized regression to eliminate features, before retraining a RandomForest on the selected features. Create a pipeline with `LassoCV` as the selector and a RandomForest as the model with the best hyperparameters found by the previous grid-search. Evaluate on the test set.

In [ ]:
# Code

## 8. [OPTIONAL]  Compare more different models

Compare more regression models with one another (for example, in addition to linear regression, decision trees, and forests, use `GradientBoostingRegressor`, and the novel `TabICLRegressor` if you have time ( https://github.com/soda-inria/tabicl ))), by performing appropriate hyperparameter tuning for each model:

In [ ]:
# Code. Search which hyper-parameters are worth tuning for each model

Choose the model that performs best and its optimal hyperparameters, then select the best subset of features for this model, and finally perform one last grid search on your model’s hyperparameters using the selected features to obtain your final model. Evaluate it on the test set.

In [ ]:
# Code

## Conclusion